# Build Hard Triplet Dataset (TF-IDF)
This notebook extracts queries from the Wikidata splits and constructs a triplet dataset `(anchor_query, positive_queries, negative_queries)` based on the following constraints:
* **Positives:** Jaccard similarity of SPARQL result sets > 0.9 **AND** Cosine similarity of TF-IDF vectors < 0.5.
* **Positives constraint:** At least 3 positive queries per anchor.
* **Negatives constraint:** At most 20 negative queries per anchor (sampled randomly from candidates with low Jaccard similarity).

In [1]:
import sys
import os
import json
import random
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(os.path.abspath('..'))
from utils import get_answers

SEMANTIC_PATH = '../../dataset/raw/jaccard_triplets_wikidata.json'
SPLITS_PATH = '../../dataset/splits/splits_wikidata.json'
CACHE_PATH = '../../cache/wikidata_answer_cache.pkl'
OUTPUT_DATASET_PATH = '../dataset/hard_triplets_tfidf_wikidata.json'

random.seed(42)

### 1. Load Data and Extract Unique Queries
Extract all unique queries from the test split to form our candidate pool.

In [2]:
# Load Data
with open(SEMANTIC_PATH, 'r') as f:
    raw_semantic_data = json.load(f)
    
with open(SPLITS_PATH, 'r') as f:
    test_indices = json.load(f).get('test_idx', [])

# Extract unique queries
unique_queries_set = set()
for idx in test_indices:
    item = raw_semantic_data[idx]
    if item.get('anchor_query'):
        unique_queries_set.add(item['anchor_query'])
        for p in item.get('positive_queries', []):
            unique_queries_set.add(p.get('query') if isinstance(p, dict) else p)

queries = list(unique_queries_set)
print(f"Loaded {len(queries)} unique queries from test split.")

Loaded 3851 unique queries from test split.


### 2. Fetch SPARQL Results
Using the cached answers, we map each query to its result set for Jaccard calculations.

In [3]:
# Fetch results for all queries (should hit the cache populated by get_answers)
query_results = {}
for q in queries:
    ans = get_answers(q, endpoint_name="wikidata") 
    query_results[q] = set(ans) if ans else set()

def calculate_jaccard(set1, set2):
    if not set1 and not set2:
        return 0.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union

### 3. Compute TF-IDF Vectors and Cosine Similarity
We use `TfidfVectorizer` to convert the raw SPARQL queries into a sparse matrix of TF-IDF features, then compute the pairwise cosine similarity.

In [4]:
# Initialize TF-IDF Vectorizer
# We use a character n-gram approach as SPARQL queries have specific syntax that words alone might miss,
# but a standard word analyzer can also work depending on your preference.
vectorizer = TfidfVectorizer(analyzer='word', token_pattern=r'(?u)\b\w+\b|\S')

# Compute TF-IDF matrix for all queries
tfidf_matrix = vectorizer.fit_transform(queries)

# Compute pairwise cosine similarity matrix
cos_sim_matrix = cosine_similarity(tfidf_matrix)
print(f"Computed TF-IDF cosine similarity matrix of shape {cos_sim_matrix.shape}")

Computed TF-IDF cosine similarity matrix of shape (3851, 3851)


### 4. Build the Hard Triplet Dataset
Filter the candidates using the 0.8 Jaccard and 0.5 TF-IDF Cosine thresholds.

In [5]:
dataset = []

for i, anchor_query in enumerate(queries):
    positives = []
    negatives = []
    
    anchor_results = query_results[anchor_query]
    
    for j, other_query in enumerate(queries):
        if i == j:
            continue
            
        other_results = query_results[other_query]
        
        # Calculate metrics
        jaccard_sim = calculate_jaccard(anchor_results, other_results)
        cosine_sim = cos_sim_matrix[i][j]
        
        # Positive condition: Highly similar results, but syntactically/lexically different
        if jaccard_sim > 0.8 and cosine_sim < 0.5:
            positives.append(other_query)
            
        # Negative condition: Low Jaccard similarity
        elif jaccard_sim < 0.2 and cosine_sim >= 0.5:
            negatives.append(other_query)
            
    # Apply constraints: At least 3 positives
    if len(positives) >= 3:
        # At most 20 negatives
        sampled_negatives = random.sample(negatives, min(len(negatives), 20))
        
        dataset.append({
            "anchor_query": anchor_query,
            "positive_queries": positives,
            "negative_queries": sampled_negatives
        })

print(f"Successfully generated {len(dataset)} hard triplets using TF-IDF.")

Successfully generated 186 hard triplets using TF-IDF.


In [6]:
# UPDATED: This will save the file right next to your notebook
OUTPUT_DATASET_PATH = '../dataset/hard_triplets_tfidf_wikidata.json'

with open(OUTPUT_DATASET_PATH, 'w') as f:
    json.dump(dataset, f, indent=4)
    
print(f"Dataset saved to {OUTPUT_DATASET_PATH}")

Dataset saved to ../dataset/hard_triplets_tfidf_wikidata.json
